# pixel-trust v4: a second dataset (Middlebury) and cross-domain transfer

Same protocol as v2 (Stable Diffusion inpainting) and v3 (GenWarp), on **Middlebury stereo 2014 + 2021**: 47 indoor scenes with dense ground-truth disparity, each used in both directions (94 pairs), split by scene group into train / calib / test (38 / 18 / 38).

1. **In-domain**: the full v2/v3 analysis on Middlebury (all signals and published baselines; semantic entropy uses SegFormer-ADE20K here).
2. **Zero-shot transfer**: trust model trained *and* calibrated on KITTI, tested on Middlebury.
3. **Recalibration**: KITTI-trained model, threshold re-fitted on n Middlebury images.

KITTI features come from the v2 and v3 notebook outputs (add them as inputs); any that are missing are regenerated here.

Settings: Accelerator **GPU T4 x2**, Internet **on**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, subprocess, sys, glob, json, time
REPO = "https://github.com/Rahul-Verma97/pixel-trust.git"
if not os.path.isdir("/kaggle/working/pixel-trust"):
    !git clone -q {REPO} /kaggle/working/pixel-trust
!pip install -q "diffusers>=0.27" "transformers>=4.45" accelerate omegaconf einops roma jaxtyping
os.chdir("/kaggle/working/pixel-trust")
sys.path.insert(0, "/kaggle/working/pixel-trust")
!git log -1 --format='%h %s'

In [ ]:
# KITTI features from earlier runs (v2 = SD inpainting, v3 = GenWarp), attached as notebook outputs
KITTI = {}
for d in sorted({os.path.dirname(os.path.dirname(f)) for f in glob.glob("/kaggle/input/**/feats/*_clean.npz", recursive=True)}):
    logs = glob.glob(os.path.join(d, "log_shard*.jsonl"))
    model = json.loads(open(logs[0]).readline()).get("model", "?") if logs else "?"
    gen = "genwarp" if "genwarp" in model.lower() else "sd"
    n = len(glob.glob(os.path.join(d, "feats", "*.npz")))
    print(f"{gen:8s} {n:4d} files  {model}  {d}")
    if n >= 400 and gen not in KITTI:
        KITTI[gen] = d
print("KITTI features found for:", sorted(KITTI))

In [ ]:
# Middlebury 2014 (23 scenes with GT) + 2021 (24 scenes): download, resize to height 368, cache
from pixel_trust.middlebury import prepare, MiddleburyDataset
MB = "/kaggle/working/middlebury"
t0 = time.time(); prepare(MB); print(f"{time.time()-t0:.0f} s")
ds = MiddleburyDataset(MB)
from collections import Counter
print(len(ds.indices()), "pairs", Counter(ds.split_of(i) for i in ds.indices()))
!rm -rf /kaggle/working/middlebury/raw

In [ ]:
import torch
NGPU = max(1, torch.cuda.device_count())
GEN_ARGS = {"sd": ["--gen", "sd", "--steps", "25"], "genwarp": ["--gen", "genwarp", "--steps", "20"]}

def stage1(dataset, root, out, gen, tag):
    procs = []
    for g in range(NGPU):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
        log = open(f"/kaggle/working/{tag}_shard{g}.log", "w")
        procs.append(subprocess.Popen([sys.executable, "run_generate.py", "--dataset", dataset, "--root", root, "--out", out,
                                       "--shard", str(g), "--nshards", str(NGPU), "--k", "4"] + GEN_ARGS[gen],
                                      stdout=log, stderr=subprocess.STDOUT, env=env))
    t0 = time.time()
    while any(p.poll() is None for p in procs):
        time.sleep(120)
        tails = [open(f"/kaggle/working/{tag}_shard{g}.log").read().strip().splitlines()[-1:] for g in range(NGPU)]
        print(f"[{tag}] {(time.time()-t0)/60:5.1f} min", tails, flush=True)
    print(f"[{tag}] exit codes:", [p.returncode for p in procs], f"{(time.time()-t0)/60:.1f} min")
    for g in range(NGPU):
        print(open(f"/kaggle/working/{tag}_shard{g}.log").read()[-1500:])

In [ ]:
# Smoke test: 2 Middlebury pairs per generator on GPU 0
import numpy as np
for gen in ("sd", "genwarp"):
    ga = " ".join(GEN_ARGS[gen])
    !CUDA_VISIBLE_DEVICES=0 python run_generate.py --dataset middlebury --root {MB} --out /kaggle/working/smoke_{gen} --limit 2 --k 4 {ga} 2>&1 | tail -6
    for f in sorted(glob.glob(f"/kaggle/working/smoke_{gen}/feats/*.npz"))[:1]:
        z = np.load(f); print(gen, f.split("/")[-1], sorted(k for k in z.files if k.startswith("f_")))

In [ ]:
# Stage 1 on Middlebury for both generators
OUT = {"sd": "/kaggle/working/mb_sd", "genwarp": "/kaggle/working/mb_genwarp"}
for gen in ("sd", "genwarp"):
    stage1("middlebury", MB, OUT[gen], gen, f"mb_{gen}")

In [ ]:
# KITTI features for any generator whose earlier output is not attached (slow: 2.6 h SD, 4.7 h GenWarp)
missing = [g for g in ("sd", "genwarp") if g not in KITTI]
if missing:
    from pixel_trust.data import find_kitti_root, download_kitti
    kroot = find_kitti_root(["/kaggle/input", "/kaggle/working/kitti"])
    if kroot is None:
        download_kitti("/kaggle/working/kitti"); kroot = find_kitti_root(["/kaggle/working/kitti"])
    for gen in missing:
        KITTI[gen] = f"/kaggle/working/kitti_{gen}"
        stage1("kitti", kroot, KITTI[gen], gen, f"kitti_{gen}")
print(KITTI)

In [ ]:
# 1. In-domain analysis on Middlebury (all signals + published baselines)
for gen in ("sd", "genwarp"):
    od = OUT[gen]
    !python run_analysis.py --out {od} > /kaggle/working/analysis_{gen}.log 2>&1; tail -2 /kaggle/working/analysis_{gen}.log
    print(open(f"{OUT[gen]}/REPORT.md").read()[:4000])

In [ ]:
# 2 + 3. Zero-shot transfer and recalibration (KITTI -> Middlebury)
pairs = f'--pair "SD inpainting={KITTI["sd"]},{OUT["sd"]}" --pair "GenWarp={KITTI["genwarp"]},{OUT["genwarp"]}"'
!python run_transfer.py {pairs} --out /kaggle/working/transfer

In [ ]:
from IPython.display import Image, display
for p in ["/kaggle/working/transfer/fig_transfer.png"] + [f"{OUT[g]}/figures/{f}.png" for g in OUT for f in ("fig_sota", "fig_examples", "fig_conformal")]:
    if os.path.exists(p):
        print(p); display(Image(p))

In [ ]:
# Package the small outputs; per-pixel features stay in the notebook output for later reuse
!cd /kaggle/working && zip -q -r pixel_trust_v4_results.zip transfer mb_sd/REPORT.md mb_sd/results.json mb_sd/figures mb_sd/png mb_sd/log_shard*.jsonl mb_genwarp/REPORT.md mb_genwarp/results.json mb_genwarp/figures mb_genwarp/png mb_genwarp/log_shard*.jsonl middlebury/index.json
!ls -la /kaggle/working/pixel_trust_v4_results.zip